# 8.1 评估方法 / Evaluation methods

怎样给模型的输出打分，自动指标和模型评委各有什么盲区，以及一个评测结果在统计上有多可靠。

How model outputs are scored, where automatic metrics and model judges are blind, and how reliable an evaluation result is statistically.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 手写 EM、F1、BLEU、ROUGE-L / EM, F1, BLEU and ROUGE-L by hand

实现四个基于字面重合的指标。测试中的数值都可以手算核对。最后一组测试用同一个参考答案比较一个措辞相同而事实错误的回答和一个措辞不同而事实正确的回答，请查看打印出的表格。

Implement four metrics based on surface overlap. All values in the tests can be checked by hand. The last group of tests compares, against one reference, an answer with the same wording and a wrong fact and an answer with other wording and the right fact. Look at the printed table.

In [ ]:
import math
import re
import string
from collections import Counter


def normalize(text):
    """The normalisation used for question-answering metrics: lower case, remove punctuation
    (string.punctuation), remove the articles a / an / the as whole words, collapse whitespace."""
    # TODO 1
    raise NotImplementedError


def exact_match(prediction, reference):
    """1.0 if the two strings are equal after normalize, else 0.0."""
    # TODO 2
    raise NotImplementedError


def token_f1(prediction, reference):
    """F1 over the normalised tokens. precision = common / len(prediction), recall = common / len(reference),
    where `common` counts shared tokens with multiplicity (Counter(p) & Counter(r)).
    If either side is empty, return 1.0 when both are empty and 0.0 otherwise."""
    # TODO 3
    raise NotImplementedError


def ngrams(tokens, n):
    """A Counter of the n-grams (as tuples) of a token list. Nothing to change here."""
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def bleu(candidate, reference, max_n=4):
    """Sentence-level BLEU with one reference and no smoothing. Tokens: lower-cased, split on whitespace.

    For n = 1..max_n the modified precision p_n = (clipped matches) / (n-grams in the candidate),
    where an n-gram of the candidate counts at most as often as it occurs in the reference.
    BLEU = BP * exp( mean of log p_n ),   BP = 1 if len(c) > len(r) else exp(1 - len(r) / len(c)).
    If any p_n is 0 (or the candidate has no n-grams of that length) the result is 0.0.
    """
    # TODO 4
    raise NotImplementedError


def lcs_length(a, b):
    """Length of the longest common subsequence of two token lists (dynamic programming)."""
    # TODO 5
    raise NotImplementedError


def rouge_l(candidate, reference):
    """ROUGE-L as an F1: precision = LCS / len(candidate), recall = LCS / len(reference). 0.0 if LCS is 0."""
    # TODO 6
    raise NotImplementedError

In [ ]:
# 检查 / checks
import math

# 1. Normalisation, exact match, F1
assert normalize("The  Sorrel Bridge!") == "sorrel bridge" and normalize("An apple, a pear.") == "apple pear"
assert normalize("theatre") == "theatre", "remove articles as whole words only"
assert exact_match("the Sorrel bridge.", "Sorrel Bridge") == 1.0 and exact_match("1754", "1974") == 0.0
assert abs(token_f1("Paris France", "Paris") - 2 / 3) < 1e-9
assert token_f1("green", "red") == 0.0 and token_f1("", "") == 1.0 and token_f1("", "x") == 0.0
assert abs(token_f1("the cat the cat", "cat") - 2 * 0.5 * 1 / 1.5) < 1e-9, "shared tokens count with multiplicity"

# 2. BLEU, checked by hand
c, r = "the cat sat on the mat", "the cat is on the mat"
assert abs(bleu(c, r, max_n=1) - 5 / 6) < 1e-9                       # 5 of 6 unigrams match
assert abs(bleu(c, r, max_n=2) - math.sqrt(5 / 6 * 3 / 5)) < 1e-9     # 3 of 5 bigrams match
assert abs(bleu("the the the the", "the cat", max_n=1) - 0.25) < 1e-9, "clipping: 'the' may count only once"
assert abs(bleu("the cat", "the cat sat on the mat", max_n=1) - math.exp(1 - 6 / 2)) < 1e-9, "brevity penalty"
assert bleu(r, r) == 1.0 and bleu("a dog", "the cat") == 0.0
assert bleu(c, r, max_n=4) == 0.0 or bleu(c, r, max_n=4) < bleu(c, r, max_n=2)

# 3. ROUGE-L
assert lcs_length("the cat sat on the mat".split(), "the cat is on the mat".split()) == 5
assert lcs_length([], ["a"]) == 0 and lcs_length(list("abcbdab"), list("bdcaba")) == 4
assert abs(rouge_l(c, r) - 5 / 6) < 1e-9 and rouge_l("x y", "a b") == 0.0

# 4. What overlap metrics see and what they miss
reference = "The Sorrel Bridge was built in 1754."
wrong = "The Sorrel Bridge was built in 1974."          # one character off, and false
right = "It dates from 1754."                           # correct, in other words
rows = [(name, exact_match(x, reference), token_f1(x, reference), bleu(x, reference, 2), rouge_l(x, reference))
        for name, x in (("wrong year, same wording", wrong), ("right year, other wording", right))]
assert rows[0][2] > 0.8 and rows[0][4] > 0.8 and rows[1][2] < 0.5 and rows[1][4] < 0.5, "overlap rewards the wording, not the fact"
print("m81_metrics: all tests passed")
print("  reference:", reference)
print("                               EM    F1    BLEU-2  ROUGE-L")
for name, em, f1, b, rl in rows:
    print(f"  {name:27s} {em:4.0f}  {f1:5.2f}  {b:6.2f}  {rl:7.2f}")

<details><summary><b>提示 1 / Hint 1</b></summary>

去冠词用 `re.sub(r'\b(a|an|the)\b', ' ', text)`，`\b` 保证只匹配完整的词。

Remove articles with `re.sub(r'\b(a|an|the)\b', ' ', text)`. The `\b` makes sure that only whole words match.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

截断后的匹配数是 `sum(min(count, ref[g]) for g, count in cand.items())`。

The clipped number of matches is `sum(min(count, ref[g]) for g, count in cand.items())`.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

最长公共子序列用二维表。`dp[i][j]` 在两个词相同时等于 `dp[i-1][j-1] + 1`，否则取 `dp[i-1][j]` 与 `dp[i][j-1]` 中较大的一个。

The longest common subsequence uses a two-dimensional table. `dp[i][j]` equals `dp[i-1][j-1] + 1` when the two words are equal and otherwise the larger of `dp[i-1][j]` and `dp[i][j-1]`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import math
import re
import string
from collections import Counter


def normalize(text):
    text = text.lower()
    text = "".join(ch for ch in text if ch not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def exact_match(prediction, reference):
    return float(normalize(prediction) == normalize(reference))


def token_f1(prediction, reference):
    p, r = normalize(prediction).split(), normalize(reference).split()
    if not p or not r:
        return float(p == r)
    common = sum((Counter(p) & Counter(r)).values())
    if common == 0:
        return 0.0
    precision, recall = common / len(p), common / len(r)
    return 2 * precision * recall / (precision + recall)


def ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def bleu(candidate, reference, max_n=4):
    c, r = candidate.lower().split(), reference.lower().split()
    log_sum = 0.0
    for n in range(1, max_n + 1):
        cand, ref = ngrams(c, n), ngrams(r, n)
        total = sum(cand.values())
        clipped = sum(min(count, ref[g]) for g, count in cand.items())
        if total == 0 or clipped == 0:
            return 0.0
        log_sum += math.log(clipped / total) / max_n
    bp = 1.0 if len(c) > len(r) else math.exp(1 - len(r) / len(c))
    return bp * math.exp(log_sum)


def lcs_length(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i, x in enumerate(a, 1):
        for j, y in enumerate(b, 1):
            dp[i][j] = dp[i - 1][j - 1] + 1 if x == y else max(dp[i - 1][j], dp[i][j - 1])
    return dp[-1][-1]


def rouge_l(candidate, reference):
    c, r = candidate.lower().split(), reference.lower().split()
    lcs = lcs_length(c, r)
    if lcs == 0:
        return 0.0
    precision, recall = lcs / len(c), lcs / len(r)
    return 2 * precision * recall / (precision + recall)
```

</details>

## 2. LLM 评委的位置偏差实验 / Position bias of an LLM judge

让一个 5 亿参数的模型在两个回答中选出正确的一个，每一对回答分别以两种顺序提交。实现读取评委对 A 与 B 的 logit 之差、统计位置偏差的四个比例，以及合并两种顺序的修正方法。

A model of 0.5 billion parameters is asked to pick the correct one of two answers, and every pair is submitted in both orders. Implement reading the difference between the logits of the judge for A and B, the four fractions that describe position bias, and the correction that combines the two orders.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def judge_prompt(question, answer_a, answer_b):
    """The text shown to the judge. Nothing to change here."""
    return (f"Question: {question}\n\nAnswer A: {answer_a}\nAnswer B: {answer_b}\n\n"
            "Which answer is correct? Reply with a single letter, A or B.")


@torch.no_grad()
def judge_margin(model, tokenizer, question, answer_a, answer_b):
    """How strongly the judge prefers A over B: logit of the token "A" minus logit of the token "B"
    at the position where the reply starts. Positive means the judge says A.

    Render judge_prompt as a user message with the chat template (add_generation_prompt=True, tokenize=False),
    tokenise, run the model once and read the logits at the last position. No generation is needed.
    """
    # TODO 1
    raise NotImplementedError


def position_stats(margins):
    """margins: one pair (m1, m2) per question.
         m1 = judge_margin with the GOOD answer shown as A,   m2 = judge_margin with the GOOD answer shown as B.
       A judge without position bias has m1 > 0 and m2 < 0.

    Returns a dict with four fractions:
      "accuracy_good_first":   share of questions with m1 > 0
      "accuracy_good_second":  share of questions with m2 < 0
      "picks_first":           share of all 2n judgements in which the judge says A
      "consistent":            share of questions where the same ANSWER wins in both orders
    """
    # TODO 2
    raise NotImplementedError


def debiased_accuracy(margins):
    """Judge every pair in both orders and combine: the good answer wins if m1 - m2 > 0.
    A constant preference for position A adds the same amount to m1 and m2 and cancels in the difference."""
    # TODO 3
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. The statistics on made-up margins
fair = [(2.0, -1.5), (0.5, -0.5), (3.0, -2.0), (1.0, -1.0)]
assert position_stats(fair) == {"accuracy_good_first": 1.0, "accuracy_good_second": 1.0, "picks_first": 0.5, "consistent": 1.0}
biased = [(4.0, 1.0), (3.0, 2.5), (5.0, 0.5), (2.0, 3.0)]            # always says A; the last pair is truly misjudged
s = position_stats(biased)
assert s == {"accuracy_good_first": 1.0, "accuracy_good_second": 0.0, "picks_first": 1.0, "consistent": 0.0}
assert debiased_accuracy(biased) == 0.75 and debiased_accuracy(fair) == 1.0

# 2. A small model as a judge. In every pair one answer is plainly right and one plainly wrong.
NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForCausalLM.from_pretrained(NAME).eval()
PAIRS = [("What is the capital of France?", "Paris is the capital of France.", "The capital of France is Lyon."),
         ("How many days are there in a week?", "A week has seven days.", "A week has ten days."),
         ("What do bees make?", "Bees make honey.", "Bees make milk."),
         ("Which planet is closest to the Sun?", "Mercury is the closest planet to the Sun.", "Jupiter is the closest planet to the Sun."),
         ("What is 12 plus 9?", "12 plus 9 is 21.", "12 plus 9 is 25."),
         ("What colour do you get by mixing blue and yellow?", "Mixing blue and yellow gives green.", "Mixing blue and yellow gives red."),
         ("In which season does snow usually fall?", "Snow usually falls in winter.", "Snow usually falls in summer."),
         ("What is the boiling point of water at sea level in degrees Celsius?", "Water boils at 100 degrees Celsius at sea level.", "Water boils at 40 degrees Celsius at sea level."),
         ("Which animal is known as man's best friend?", "The dog is known as man's best friend.", "The shark is known as man's best friend."),
         ("How many legs does a spider have?", "A spider has eight legs.", "A spider has five legs."),
         ("What is the opposite of hot?", "The opposite of hot is cold.", "The opposite of hot is loud."),
         ("Which ocean lies between Europe and America?", "The Atlantic Ocean lies between Europe and America.", "The Indian Ocean lies between Europe and America.")]
m = judge_margin(model, tok, *PAIRS[0])
assert isinstance(m, float)
margins = [(judge_margin(model, tok, q, good, bad), judge_margin(model, tok, q, bad, good)) for q, good, bad in PAIRS]
s = position_stats(margins)
d = debiased_accuracy(margins)
assert s["picks_first"] >= 0.8, "this small judge should show a strong preference for the first position"
assert s["accuracy_good_first"] - s["accuracy_good_second"] >= 0.5, "its verdict should depend heavily on the order"
assert d >= s["accuracy_good_second"]

print("m81_judge: all tests passed")
print(f"  good answer shown first:  judged correctly in {s['accuracy_good_first']:.0%} of 12 pairs")
print(f"  good answer shown second: judged correctly in {s['accuracy_good_second']:.0%}")
print(f"  says A in {s['picks_first']:.0%} of 24 judgements; same answer wins in both orders for {s['consistent']:.0%} of pairs")
print(f"  combining both orders (m1 - m2 > 0): correct in {d:.0%}")

<details><summary><b>提示 1 / Hint 1</b></summary>

与 7.1 的做法相同，渲染对话模板后做一次前向计算，读取最后一个位置的 logits，不需要生成。

As in 7.1, render the chat template, run one forward pass and read the logits at the last position. No generation is needed.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

同一个回答在两种顺序下都胜出，等价于 `(m1 > 0) == (m2 < 0)`。

The same answer winning in both orders is equivalent to `(m1 > 0) == (m2 < 0)`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def judge_prompt(question, answer_a, answer_b):
    return (f"Question: {question}\n\nAnswer A: {answer_a}\nAnswer B: {answer_b}\n\n"
            "Which answer is correct? Reply with a single letter, A or B.")


@torch.no_grad()
def judge_margin(model, tokenizer, question, answer_a, answer_b):
    text = tokenizer.apply_chat_template([{"role": "user", "content": judge_prompt(question, answer_a, answer_b)}],
                                         add_generation_prompt=True, tokenize=False)
    logits = model(tokenizer(text, return_tensors="pt").input_ids).logits[0, -1]
    return (logits[tokenizer("A").input_ids[0]] - logits[tokenizer("B").input_ids[0]]).item()


def position_stats(margins):
    n = len(margins)
    return {"accuracy_good_first": sum(m1 > 0 for m1, _ in margins) / n,
            "accuracy_good_second": sum(m2 < 0 for _, m2 in margins) / n,
            "picks_first": sum((m1 > 0) + (m2 > 0) for m1, m2 in margins) / (2 * n),
            "consistent": sum((m1 > 0) == (m2 < 0) for m1, m2 in margins) / n}


def debiased_accuracy(margins):
    return sum((m1 - m2) > 0 for m1, m2 in margins) / len(margins)
```

</details>

## 3. Cohen's kappa 与 bootstrap 置信区间 / Cohen's kappa and bootstrap confidence intervals

实现 Cohen's kappa、均值的自助法置信区间，以及两个系统的配对自助比较。测试会把 kappa 与 scikit-learn 的结果核对，并比较 8、20 和 1000 个样本时置信区间的宽度。

Implement Cohen's kappa, the bootstrap confidence interval of a mean, and the paired bootstrap comparison of two systems. The tests check kappa against scikit-learn and compare the width of the confidence interval for 8, 20 and 1000 samples.

In [ ]:
import random


def cohen_kappa(a, b):
    """Agreement between two raters beyond what chance would give.

    a, b: two equally long lists of labels for the same items.
      p_o = observed share of items on which the raters agree
      p_e = sum over the labels c of (share of c in a) * (share of c in b)       agreement expected by chance
      kappa = (p_o - p_e) / (1 - p_e)          (return 1.0 if p_e == 1)
    """
    # TODO 1
    raise NotImplementedError


def bootstrap_ci(values, n_resamples=2000, level=0.95, seed=0):
    """A confidence interval for the mean of `values` by the percentile bootstrap.

    Use rng = random.Random(seed). Repeat n_resamples times: draw len(values) items WITH replacement
    (rng.choice) and take their mean. Sort the means and return
        (means[int((1 - level) / 2 * n_resamples)],  means[int((1 + level) / 2 * n_resamples) - 1])
    """
    # TODO 2
    raise NotImplementedError


def paired_bootstrap(scores_a, scores_b, n_resamples=2000, seed=0):
    """Two systems scored on the SAME items. In what share of resamples is the total of A above the total of B?

    Use rng = random.Random(seed). Each resample draws n item indices with replacement (rng.randrange(n))
    and uses the same indices for both systems, which keeps the pairing.
    A value near 1 means A is reliably better on this kind of item; a value near 0.5 means the data cannot tell.
    """
    # TODO 3
    raise NotImplementedError

In [ ]:
# 检查 / checks
from sklearn.metrics import cohen_kappa_score

# 1. Kappa, by hand and against scikit-learn
a = ["y", "y", "y", "y", "n", "n", "n", "n", "y", "n"]
b = ["y", "y", "y", "n", "n", "n", "n", "y", "y", "n"]
assert abs(cohen_kappa(a, b) - 0.6) < 1e-9                      # p_o = 0.8, p_e = 0.5
assert cohen_kappa(a, a) == 1.0
skew_a = ["ok"] * 18 + ["bad", "ok"]
skew_b = ["ok"] * 18 + ["ok", "bad"]
k = cohen_kappa(skew_a, skew_b)
assert sum(x == y for x, y in zip(skew_a, skew_b)) / 20 == 0.9 and k < 0.01, "90% raw agreement, and no agreement beyond chance"
three_a, three_b = list("aabbccabca"), list("abbbccacca")
for x, y in ((a, b), (skew_a, skew_b), (three_a, three_b)):
    assert abs(cohen_kappa(x, y) - cohen_kappa_score(x, y)) < 1e-9

# 2. Bootstrap confidence intervals
few = [1] * 16 + [0] * 4                                        # 16 of 20 correct
lo, hi = bootstrap_ci(few)
assert 0.55 <= lo <= 0.68 and 0.92 <= hi <= 1.0, (lo, hi)
many = [1] * 800 + [0] * 200                                    # the same 80%, on 1000 items
lo2, hi2 = bootstrap_ci(many)
assert 0.76 <= lo2 <= 0.79 and 0.81 <= hi2 <= 0.84
assert (hi - lo) > 5 * (hi2 - lo2), "fifty times the data, about seven times narrower"
assert bootstrap_ci(few, seed=1) == bootstrap_ci(few, seed=1), "the same seed must give the same interval"
lo8, hi8 = bootstrap_ci([1, 1, 1, 1, 0, 0, 0, 0])              # 4 of 8: the sample size of several experiments in this roadmap
assert hi8 - lo8 >= 0.6

# 3. Comparing two systems on the same items
A = [1] * 16 + [0] * 4
B = [1] * 14 + [0] * 6
p_small = paired_bootstrap(A, B)
assert 0.6 < p_small < 0.99, "16 against 14 of 20 is suggestive, not settled"
A_big, B_big = A * 25, B * 25                                   # the same rates on 500 items
assert paired_bootstrap(A_big, B_big) > 0.999
assert abs(paired_bootstrap(A, A) - 0.0) < 1e-9, "a system is never strictly better than itself"

print("m81_stats: all tests passed")
print(f"  kappa with 90% raw agreement on a skewed label distribution: {k:.2f}")
print(f"  95% interval for 16 of 20 correct:   [{lo:.2f}, {hi:.2f}]")
print(f"  95% interval for 800 of 1000 correct: [{lo2:.2f}, {hi2:.2f}]")
print(f"  95% interval for 4 of 8 correct:     [{lo8:.2f}, {hi8:.2f}]")
print(f"  16 vs 14 of 20 on the same items: A ahead in {p_small:.0%} of resamples")

<details><summary><b>提示 1 / Hint 1</b></summary>

偶然一致率是对所有标签求和，`(a.count(c) / n) * (b.count(c) / n)`。

Chance agreement sums over all labels, `(a.count(c) / n) * (b.count(c) / n)`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

重抽样用 `rng.choice(values)` 抽 `len(values)` 次，注意是有放回的。

Resampling draws `len(values)` times with `rng.choice(values)`. Note that this is with replacement.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

配对比较中，每次重抽样先抽出一组下标，两个系统都用这一组下标。

In the paired comparison every resample first draws one set of indices, which both systems use.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import random


def cohen_kappa(a, b):
    n = len(a)
    labels = set(a) | set(b)
    p_o = sum(x == y for x, y in zip(a, b)) / n
    p_e = sum((a.count(c) / n) * (b.count(c) / n) for c in labels)
    if p_e == 1:
        return 1.0
    return (p_o - p_e) / (1 - p_e)


def bootstrap_ci(values, n_resamples=2000, level=0.95, seed=0):
    rng = random.Random(seed)
    n = len(values)
    means = sorted(sum(rng.choice(values) for _ in range(n)) / n for _ in range(n_resamples))
    lo = means[int((1 - level) / 2 * n_resamples)]
    hi = means[int((1 + level) / 2 * n_resamples) - 1]
    return lo, hi


def paired_bootstrap(scores_a, scores_b, n_resamples=2000, seed=0):
    rng = random.Random(seed)
    n = len(scores_a)
    wins = 0
    for _ in range(n_resamples):
        idx = [rng.randrange(n) for _ in range(n)]
        wins += sum(scores_a[i] for i in idx) > sum(scores_b[i] for i in idx)
    return wins / n_resamples
```

</details>

## 4. 手算 kappa 与置信区间 / Kappa and confidence intervals by hand

在纸上完成，可以使用计算器。
(a) 两位标注者对 50 条回答标注“正确”或“错误”。两人都标为正确的有 30 条，都标为错误的有 10 条，甲标正确而乙标错误的有 6 条，甲标错误而乙标正确的有 4 条。计算 $p_o$、$p_e$ 和 $\kappa$。
(b) 一个模型在 200 道题上的正确率是 0.75。用正态近似计算 95% 置信区间。
(c) 另一个模型在同样的 200 道题上的正确率是 0.78。仅凭这两个数字，能否认为第二个模型更好？还需要什么信息？
(d) 一个评委的读数满足 $m_1 = s + b$、$m_2 = -s + b$。测得 $m_1 = 3.2$、$m_2 = 1.8$。求 $s$ 和 $b$，并说明评委在两种顺序下各选了哪个位置、它对内容的判断是什么。

Work on paper. A calculator may be used.
(a) Two annotators label 50 answers as correct or wrong. Both say correct for 30, both say wrong for 10, the first says correct and the second wrong for 6, and the first says wrong and the second correct for 4. Compute $p_o$, $p_e$ and $\kappa$.
(b) A model has an accuracy of 0.75 on 200 questions. Compute the 95% confidence interval with the normal approximation.
(c) Another model has an accuracy of 0.78 on the same 200 questions. From these two numbers alone, can the second model be considered better? What further information is needed?
(d) The readings of a judge satisfy $m_1 = s + b$ and $m_2 = -s + b$. Measured are $m_1 = 3.2$ and $m_2 = 1.8$. Find $s$ and $b$ and say which position the judge chose in each order and what its judgement of the content is.

<details><summary><b>提示 1 / Hint 1</b></summary>

(a) 甲标正确的比例是 36/50，乙标正确的比例是 34/50。

(a) The first annotator says correct for 36/50 and the second for 34/50.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(b) $\text{SE} = \sqrt{p(1-p)/n}$。

(b) $\text{SE} = \sqrt{p(1-p)/n}$.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) $p_o = (30 + 10)/50 = 0.8$。甲标正确的比例为 0.72，乙为 0.68。$p_e = 0.72 \times 0.68 + 0.28 \times 0.32 = 0.4896 + 0.0896 = 0.5792$。$\kappa = (0.8 - 0.5792)/(1 - 0.5792) \approx 0.525$。

(b) $\text{SE} = \sqrt{0.75 \times 0.25 / 200} \approx 0.0306$。区间为 $0.75 \pm 1.96 \times 0.0306$，即约 0.69 到 0.81。

(c) 不能。3 个百分点的差别小于单个正确率的区间半宽 6 个百分点。由于两个模型是在同一批题目上测试的，应当做配对比较，这需要知道每道题上两个模型各自的对错，例如只有第一个模型答对的题数和只有第二个模型答对的题数。如果两者在绝大多数题目上结果相同，只在少数题目上有差别，配对比较可能给出比单独看两个区间更明确的结论。

(d) 两式相加得 $2b = 5.0$，$b = 2.5$。两式相减得 $2s = 1.4$，$s = 0.7$。两种顺序下 $m$ 都为正，评委都选了位置 A，表面上看它在正确答案位于 B 时判错了。扣除位置偏好后 $s > 0$，说明它对内容的判断是偏向正确答案的，只是这个倾向比位置偏好弱得多。

---

(a) $p_o = (30 + 10)/50 = 0.8$. The first annotator says correct for 0.72 and the second for 0.68. $p_e = 0.72 \times 0.68 + 0.28 \times 0.32 = 0.4896 + 0.0896 = 0.5792$. $\kappa = (0.8 - 0.5792)/(1 - 0.5792) \approx 0.525$.

(b) $\text{SE} = \sqrt{0.75 \times 0.25 / 200} \approx 0.0306$. The interval is $0.75 \pm 1.96 \times 0.0306$, about 0.69 to 0.81.

(c) No. A difference of 3 percentage points is smaller than the half-width of 6 points of the interval of a single accuracy. Since the two models were tested on the same questions, a paired comparison is called for, which requires knowing for every question whether each model was right, for example the number of questions only the first model answered correctly and the number only the second did. If the two agree on nearly all questions and differ on a few, the paired comparison may give a clearer conclusion than the two intervals viewed separately.

(d) Adding the two equations gives $2b = 5.0$, so $b = 2.5$. Subtracting gives $2s = 1.4$, so $s = 0.7$. $m$ is positive in both orders, so the judge chose position A both times and, on the surface, judged wrongly when the right answer was at B. With the position preference removed, $s > 0$ shows that its judgement of the content leans towards the right answer, only far more weakly than its preference for a position.

</details>